<a href="https://colab.research.google.com/github/AlejoCNYT/CEIA-AMIA-28Co2026-TP-Final/blob/main/AMIA_2026_TP_5_fantasticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 1: LDA/QDA y optimización matemática de modelos

# CEIA 28Co2026 - Facultad de Ingeniería - UBA
# Análisis matemático para IA - AMIA
## Trabajo Práctico Final

### Integrantes

- Daniel Alejandro Acero Varela
- Nicolás Bolé
- Tomas Moretti
- Leticia L. Rodríguez
- Juan Ignacio Trentalance

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.


# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [ ]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [ ]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    #print("log_a_priori:", modelo.log_a_priori)
    #print(a_priori)
    # Q3: para que sirve bincount? Rta: Cuantas veces aparece un entero en un array
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?
    # Rta: pq la implementación de _fit_params (que no esta implementada en esta clase abstracta
    # va a terminar usando la variable de clase log_a_priori

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [ ]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Rta: Porque se usa una máscara booleana, que es ese igualdad. La máscara idx es un vector
    # unidimensional (n,) e y era de la forma (n,1).

    # Q6: por que se usa bias=True en vez del default bias=False?
    # Acorde con la documentación de numpy, np.cov el parametro bias es la normalización n-1.
    # Con bias en True, la normalización es sobre N. Raro pq son muestras y la cov define N-1.
    # Tiene que ver con las asumpciones de QDA depende si es MLE (N) o un estimador unbiassed (n-1)
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?
    # Porque sacamos la media de las columnas (features), no de la fila (muestras), por eso axis=1.

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [ ]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x # calcula inner_prod[j]=(x−μj​)TΣj−1​(x−μj​)

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten() # det la inversa de covarianza, plano

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [ ]:
class FasterQDA(TensorizedQDA):

    def _predict_log_conditionals_all(self, X):
        unbiased_X = X - self.tensor_means
        full = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X
        # Esto paraleliza sobre N
        inner_prod = np.diagonal(full, axis1=1, axis2=2)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))
        return log_det[:, None] - 0.5 * inner_prod


    def predict(self, X):
      log_post = self.log_a_priori[:, None] + self._predict_log_conditionals_all(X)
      return np.argmax(log_post, axis=0).reshape(1, -1)

In [ ]:
class EfficientQDA(FasterQDA):

    def _predict_log_conditionals_all(self, X):
        unbiased_X = X - self.tensor_means
        A = self.tensor_inv_cov @ unbiased_X

        inner_prod = np.sum(unbiased_X * A, axis=1)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))
        return log_det[:, None] - 0.5 * inner_prod

In [ ]:

# Para poder aplicar la descomposición de Cholesky ($A = L L^T$),
# una matriz cuadrada $A$ de números reales debe cumplir estrictamente con dos condiciones:
# Ser simétrica: La matriz debe ser igual a su transpuesta ($A = A^T$).
# Esto significa que los elementos reflejados a través de la diagonal principal son idénticos.
# Ser definida positiva: Para cualquier vector real no nulo $x$, el producto $x^T A x$ debe ser mayor a cero ($x^T A x > 0$).
# En la práctica, la forma más común de confirmar esto es verificando que todos los autovalores (eigenvalues) de la matriz sean estrictamente positivos,
# o utilizando el criterio de Sylvester (donde todos los determinantes de las submatrices principales, o menores, deben ser mayores a cero).
class QDA_Chol1(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.L_invs = [LA.inv(cholesky(np.cov(X[:, y.flatten()==idx], bias=True), lower=True))
                       for idx in range(len(self.log_a_priori))]
        self.means = [X[:, y.flatten()==idx].mean(axis=1, keepdims=True)
                      for idx in range(len(self.log_a_priori))]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        z = L_inv @ (x - self.means[class_idx])
        return np.log(np.diag(L_inv)).sum() - 0.5 * np.sum(z**2)

In [ ]:
class QDA_Chol2(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.Ls = [cholesky(np.cov(X[:, y.flatten()==idx], bias=True), lower=True)
                   for idx in range(len(self.log_a_priori))]
        self.means = [X[:, y.flatten()==idx].mean(axis=1, keepdims=True)
                      for idx in range(len(self.log_a_priori))]

    def _predict_log_conditional(self, x, class_idx):
        L = self.Ls[class_idx]
        z = solve_triangular(L, x - self.means[class_idx], lower=True)
        return -np.log(np.diag(L)).sum() - 0.5 * np.sum(z**2)

In [ ]:
class QDA_Chol3(BaseBayesianClassifier):
    def _fit_params(self, X, y):
        self.L_invs = [dtrtri(cholesky(np.cov(X[:, y.flatten()==idx], bias=True), lower=True), lower=1)[0]
                       for idx in range(len(self.log_a_priori))]
        self.means = [X[:, y.flatten()==idx].mean(axis=1, keepdims=True)
                      for idx in range(len(self.log_a_priori))]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        z = L_inv @ (x - self.means[class_idx])
        return np.log(np.diag(L_inv)).sum() - 0.5 * np.sum(z**2)


        '''#Conclusiones x team:
        La diferencia principal radica en la especialización y la eficiencia:
        LA.inv (ya sea de NumPy o SciPy) es un algoritmo genérico diseñado para
        invertir cualquier tipo de matriz, mientras que dtrtri está optimizado única
        y exclusivamente para matrices triangulares.
        O(n³) con la mitad de operaciones (Aprovecha los ceros).'''





In [ ]:
class TensorizedCholQDA(QDA_Chol1):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_invs = np.stack(self.L_invs)
        self.tensor_means = np.stack(self.means)
        self.log_det_terms = np.log(np.diagonal(self.tensor_L_invs, axis1=1, axis2=2)).sum(axis=1)

    def _predict_log_conditionals(self, X):
        z = self.tensor_L_invs @ (X[None, :, :] - self.tensor_means[:, :, :])
        return self.log_det_terms[:, None] - 0.5 * np.sum(z**2, axis=1)

    def predict(self, X):
        return np.argmax(self.log_a_priori[:, None] + self._predict_log_conditionals(X), axis=0).reshape(1, -1)

        #el reshape 1,-1 es convertir un vector Rn, en una fila 1xn.


In [ ]:
class EfficientCholQDA(QDA_Chol3):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_invs = np.stack(self.L_invs)
        self.tensor_means = np.stack(self.means)
        self.log_det_terms = np.log(np.diagonal(self.tensor_L_invs, axis1=1, axis2=2)).sum(axis=1)

    def predict(self, X):
        z = self.tensor_L_invs @ (X[None, :, :] - self.tensor_means[:, :, :])
        log_cond = self.log_det_terms[:, None] - 0.5 * np.sum(z**2, axis=1)
        return np.argmax(self.log_a_priori[:, None] + log_cond, axis=0).reshape(1, -1)


## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [ ]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])


  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [ ]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [ ]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [ ]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5], print(X_full[:5])

[[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
  2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 1.120e+01 1.000e+02 2.650e+00 2.760e+00
  2.600e-01 1.280e+00 4.380e+00 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 1.860e+01 1.010e+02 2.800e+00 3.240e+00
  3.000e-01 2.810e+00 5.680e+00 1.030e+00 3.170e+00 1.185e+03]
 [1.437e+01 1.950e+00 2.500e+00 1.680e+01 1.130e+02 3.850e+00 3.490e+00
  2.400e-01 2.180e+00 7.800e+00 8.600e-01 3.450e+00 1.480e+03]
 [1.324e+01 2.590e+00 2.870e+00 2.100e+01 1.180e+02 2.800e+00 2.690e+00
  3.900e-01 1.820e+00 4.320e+00 1.040e+00 2.930e+00 7.350e+02]]


(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]),
 None)

In [ ]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [ ]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
to_bench = [QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedCholQDA, EfficientCholQDA]

for model in to_bench:
    b.bench(model)

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedCholQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedCholQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientCholQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientCholQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

# shape inv_cov = (3, 13, 13)
# tensor_media (3, 13, 1)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# A REVISAR !
b.bench(EfficientQDA)

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# A REVISAR !
b.bench(FasterQDA)

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.393614,0.186105,3.125692,0.665612,0.982407,0.018981,0.004145,0.008420,0.004917
QDA_Chol1,0.454095,0.263348,2.409505,0.473261,0.982593,0.018715,0.000658,0.008171,0.000072
QDA_Chol2,0.373122,0.140296,4.869248,0.376073,0.985741,0.017899,0.000917,0.007969,0.000131
QDA_Chol3,0.355687,0.102483,2.513220,0.183210,0.983333,0.018175,0.000677,0.007950,0.000269
TensorizedCholQDA,0.537540,0.152085,0.076439,0.039041,0.986111,0.018219,0.000875,0.060539,0.000102
EfficientCholQDA,0.392681,0.060034,0.065062,0.022562,0.982222,0.018147,0.000629,0.060196,0.000113
TensorizedQDA,0.417147,0.085319,1.447535,0.128967,0.984444,0.018181,0.000865,0.012222,0.000220
EfficientQDA,0.430879,0.100968,0.099434,0.031331,0.986481,0.018181,0.000880,0.076370,0.000252
FasterQDA,0.438218,0.133059,0.109848,0.052808,0.985000,0.018478,0.000913,0.109337,0.000238


In [ ]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil

summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.393614,3.125692,0.982407
QDA_Chol1,0.454095,2.409505,0.982593
QDA_Chol2,0.373122,4.869248,0.985741
QDA_Chol3,0.355687,2.513220,0.983333
TensorizedCholQDA,0.537540,0.076439,0.986111
EfficientCholQDA,0.392681,0.065062,0.982222
TensorizedQDA,0.417147,1.447535,0.984444
EfficientQDA,0.430879,0.099434,0.986481
FasterQDA,0.438218,0.109848,0.985000


In [ ]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.393614,0.186105,3.125692,0.665612,0.982407,0.018981,0.004145,0.008420,0.004917,1.000000,1.000000,1.000000,1.000000
QDA_Chol1,0.454095,0.263348,2.409505,0.473261,0.982593,0.018715,0.000658,0.008171,0.000072,0.866809,1.297234,1.014243,1.030521
QDA_Chol2,0.373122,0.140296,4.869248,0.376073,0.985741,0.017899,0.000917,0.007969,0.000131,1.054919,0.641925,1.060502,1.056666
QDA_Chol3,0.355687,0.102483,2.513220,0.183210,0.983333,0.018175,0.000677,0.007950,0.000269,1.106629,1.243700,1.044365,1.059201
TensorizedCholQDA,0.537540,0.152085,0.076439,0.039041,0.986111,0.018219,0.000875,0.060539,0.000102,0.732250,40.891318,1.041850,0.139091
EfficientCholQDA,0.392681,0.060034,0.065062,0.022562,0.982222,0.018147,0.000629,0.060196,0.000113,1.002376,48.041735,1.046011,0.139884
TensorizedQDA,0.417147,0.085319,1.447535,0.128967,0.984444,0.018181,0.000865,0.012222,0.000220,0.943585,2.159320,1.044036,0.688944
EfficientQDA,0.430879,0.100968,0.099434,0.031331,0.986481,0.018181,0.000880,0.076370,0.000252,0.913514,31.434837,1.044036,0.110258
FasterQDA,0.438218,0.133059,0.109848,0.052808,0.985000,0.018478,0.000913,0.109337,0.000238,0.898214,28.454826,1.027224,0.077014


In [ ]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.393614,3.125692,0.982407,1.000000,1.000000,1.000000,1.000000
QDA_Chol1,0.454095,2.409505,0.982593,0.866809,1.297234,1.014243,1.030521
QDA_Chol2,0.373122,4.869248,0.985741,1.054919,0.641925,1.060502,1.056666
QDA_Chol3,0.355687,2.513220,0.983333,1.106629,1.243700,1.044365,1.059201
TensorizedCholQDA,0.537540,0.076439,0.986111,0.732250,40.891318,1.041850,0.139091
EfficientCholQDA,0.392681,0.065062,0.982222,1.002376,48.041735,1.046011,0.139884
TensorizedQDA,0.417147,1.447535,0.984444,0.943585,2.159320,1.044036,0.688944
EfficientQDA,0.430879,0.099434,0.986481,0.913514,31.434837,1.044036,0.110258
FasterQDA,0.438218,0.109848,0.985000,0.898214,28.454826,1.027224,0.077014


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

RTA: **TensorizedQDA** paraleliza exclusivamente sobre las $k$ clases. Sigue incurriendo en un ciclo for para predecir las observaciones secuencialmente a través del uso heredado de self._predict_one(x) llamado dentro del método base predict.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

RTA: Los shapes observados son:
                 inv_cov = (3, 13, 13)
                 tensor_media (3, 13, 1)

Estos shapes son de 3 dimensiones ya que se apilan las k matrices de coviaranzas de pxp.

Se llega al mismo resultado ya que la tensorizacion apila las matrices de covarianza y se realizan calculos en paralelo ( en el caso de QDA esto es seriado por el loop for ).

```    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)
```

Extendiendo la respuesta, para el dataset de vinos, observamos la dimensión del tensor inv_cov. Su primer componente tiene el valor 3 y representa las 3 clases de vino. Luego, para cada clase hay una matriz de 13 x 13, donde 13 es cada uno de los features del dataset. Sus componentes representan la inversa de la matriz de covarianza para cada clase.

Para el caso de tensor_media, la representación es similar pero la última componente, de tamaño 1, trata de un único valor correspondiente a la media de cada uno de los 13 features para cada una de las 3 clases.

Llega al mismo resultado que qda, porque tensorized llama a los entrenamientos de qda(super_fit params) . Tambien hace la misma cuenta (el log) que qda lo hace para cada clase. Solo cambia en como se ejecuta, qda lo hace en un loop. El loop de las n observsaciones lo sigue haciendo.


### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

`FasterQDA` elimina el loop o ciclo que había sobre las observaciones que hacía `TensorizedQDA`. Se observa que `FasterQDA` no llama al predict_one que recorre observación por observación, sino que aprovecha las ventajas de la libería de numpy para el procesamiento n-dimensional optimizado. Esto quedó implementado en el método predict definido:

```
def predict(self, X):
    log_post = self.log_a_priori[:, None] + self._predict_log_conditionals_all(X)
    return np.argmax(log_post, axis=0).reshape(1, -1)"
```

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

La matriz nxn aparece en `FasterQDA` en el método `predict_log_conditionals_all` en la variable full. Definida de la siguiente forma:

```
full = unbiased_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_X
```

5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.


Demostración:

Definimos $A \in \mathbb{R}^{n \times p}$  y $B \in \mathbb{R}^{p \times n}$, dado que la multiplicación de matrices esta dada por una matriz $C$ de  $n \times n$ que se calcula de la siguiente forma:

$$C_{ij} = \sum_{k=1}^{p} A_{ik} B_{kj}$$

Usamos estas dimensiones para obtener una matriz $C$ cuadrada, pero entendemos que la multiplicación podría resultar no cuadrada.

La $diag$ es extrare los elementos de la matriz y agruparlos en un vector.
Por ejemplo, si $C$ es matriz de 3x3, $diag(C)$ esta definido:

$$C = \begin{pmatrix}  C_{11} & C_{12} & C_{13} \\  C_{21} & C_{22} & C_{23} \\  C_{31} & C_{32} & C_{33}  \end{pmatrix}$$

$$diag(C) =  [C_{11}, C_{22}, C_{33}]$$

Podemos escribir su forma genérica como:

$$\text{diag}(C) = [C_{ii}] \quad \forall i \in \{1, 2, \dots, n\}$$

Ahora si, yendo a la demostración, arrancamos por:

$$diag(A \cdot B) = $$

Y usando las definiciones de arriba vemos que:

$$diag(A \cdot B) = [(A \cdot B)_{ii}] =  \sum_{k=1}^{p} A_{ik} B_{ki}$$

Por definición de matriz transpuesta, sabemos que el elemento en la fila $k$ y columna $i$ de $B$ es exactamente el elemento en la fila $i$ y columna $k$ de $B^T$:$$B_{ki} = (B^T)_{ik}$$

$$diag(A \cdot B) = [(A \cdot B)_{ii}] = \sum_{k=1}^{p} A_{ik} (B^T)_{ik}$$

La definición del producto escalar es:

Sean $A, B \in \mathbb{R}^{m \times n}$,

$$C_{ij} = (A \odot B)_{ij} = A_{ij} \cdot B_{ij} \quad \forall i \in \{1, \dots, m\}, \forall j \in \{1, \dots, n\}$$

Entonces, si reemplaamozas por este producto:

$$\forall i, \quad 1 \leq i \leq n : [diag(A \cdot B)]_{i} =  (A \cdot B)_{ii} = \sum_{k=1}^{p} (A \odot B^T)_{ik} $$

$p$ es la cantidad de columnas que tenia la matriz A, entonces en numpy usamos axis = 1 para indicar las columnas, podemos concluir que es el producto interno sumado a travez de la columna.

$$[diag(A \cdot B)]_{i} =  (A \cdot B)_{ii} = \sum_{k=1}^{p} (A \odot B^T)_{ik} = np.sum(A \odot B^T, axis=1) $$


6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?



rta:   En los chequeos que se ven de los summary FasterQDA es mas rapida que Tensorised, y la version EfficientQDA es aun un poco mas rapida. Se condice con lo esperado, porque python no es rapido para los loops pero si para operaciones matriciales.



## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

        
**Demostración:**  
Si $A$ es simétrica definida positiva y $A = L L^T$, donde $L$ es triangular inferior no singular:
$$A^{-1} = (L L^T)^{-1} = (L^T)^{-1} L^{-1} = L^{-T} L^{-1}$$

Ahora, para aquel vector que es centrado $u = x - \mu_j$, la forma cuadrática resulta ser:
$$(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j) = u^T (L_j^{-T} L_j^{-1}) u = (L_j^{-1} u)^T (L_j^{-1} u)$$

Al definir la denominada variable transformada $z = L_j^{-1} u = L_j^{-1} (x - \mu_j)$:
$$(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j) = z^T z = \Vert{}z\Vert{}_2^2 = \sum_{i=1}^p z_i^2$$

Su utilidad recae en:
1. *Evitar la inversión explícita densa:* No se puede calcular el $\Sigma^{-1}$ de forma directa ($O(p^3)$ es costoso y numéricamente también es inestable ante determinantes pequeños).
2. *Cálculo trivial del determinante:* Tenemos que $L$ es triangular, $\vert{}A\vert{} = \vert{}L\vert{}^2 = \left(\prod \text{diag}(L)\right)^2$, de donde:
$$\log \vert{}\Sigma_j^{-1}\vert{} = \log \vert{}L_j^{-T} L_j^{-1}\vert{} = 2 \log \vert{}L_j^{-1}\vert{} = 2 \sum_{i=1}^p \log((L_j^{-1})_{ii})$$
Al reducir el cómputo de la log-verosimilitud en sumatorias de logaritmos de la diagonal y normas euclídeas de $z$.

*RTA*: Cholesky nos permite calcular cuadraticamente el QDA sin calcular explícitamente \(A^{-1}\), lo cual generalemnte es más eficiente y más estable numéricamente.


7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

*RTA*: el `QDA` calcula e invierte la matriz de covarianza completa mediante la línea `np.linalg.inv`, evaluandola de manera  cuadrática $(x-\mu)^T \Sigma^{-1} (x-\mu)$ con productos entre matrices densos y determinantes que usan la función `np.linalg.det`. Por su parte, `QDA_Chol1` descompone la llamada covarianza muestral $\Sigma_j = L_j L_j^T$ con `scipy.linalg.cholesky(..., lower=True)` y la almacena cuando se realiza el cálculo de la inversa triangular $L_j^{-1} = \text{inv}(L_j)$. A continuación se detalla el paso a paso:

1. Para cada una de las observaciones $x$ y clase $j$, se obtiene la desviación de la media  $u = x - \mu_j$
2. Se proyecta el residuo usando la matriz inversa triangular previamente obtenida: $z = L_j^{-1} u$
3. Se hace el c[alculo de la norma cuadrática del vector resultante ($\Vert{}z\Vert{}^2 = \sum z_i^2$)
4. Se hace la sumatoria de los elementos diagonales de $L_j^{-1}$ = $\sum \log(\text{diag}(L_j^{-1}))$.
5. Se le ensambla la log-probabilidad condicional de esta manera:
$$\log P(x \mid G=j) \propto \sum \log(\text{diag}(L_j^{-1})) - \frac{1}{2} \sum z_i^2$$
6. Por último, sumamos el prior $\log \pi_j$ y predecimos $\arg\max_j \{\log P(G=j \mid X=x)\}$.

8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

| Variante | Qué se almacena en el `fit` | Costo de `fit` ($O$) | Estrategia `predict` | Costo por observación ($O$) |
|---|---|---|---|---|
| **`QDA_Chol1`** | $L^{-1}$ | $O(p^3)$ — Cholesky + `LA.inv` genérico | Multiplicación directa $z = L^{-1} u$ | $O(p^2)$ |
| **`QDA_Chol2`** | $L$ (sin invertir) | $O(p^3)$ — Solo Cholesky | Resuelve sistema triangular $L z = u$ con `solve_triangular` | $O(p^2)$ |
| **`QDA_Chol3`** | $L^{-1}$ | $O(p^3)$ — Cholesky + `dtrtri` (LAPACK triangular) | Multiplicación directa $z = L^{-1} u$ | $O(p^2)$ |

DE lo cual podemos decir que:
* `QDA_Chol2` se encarga de ahorrar trabajo en el entrenamiento pero penaliza de manera significativa al test, puesto que `solve_triangular` genera mayor carga a las llamadas secuenciales en Python.
* `QDA_Chol3` es la más óptima de las 3 no vectorizadas ya que  utiliza `dtrtri` de LAPACK que hace uso de la estructura triangular, lo cual ahorra la mitad de las operaciones de inverción de entrada, dejando listo $L^{-1}$ ágilmente.

9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?


   Vemos que `FasterQDA` con ~0.11 ms, y `EfficientQDA` de ~0.10 ms, son un orden de magnitud más rápidas en predicción, si recordamos speedup de $\approx 28\times$ - $31\times$ vs un baseline `QDA` de 3.13 ms, eliminando completamebte el bucle `for` de las observaciones. De otra parte, pudimos notar que `QDA_Chol2` tiene peor tiempo test de todas, ~4.87 ms, con speedup de $0.64\times$. Al resolver una sustitución hacia adelante (`solve_triangular`) para cada una de las observaciones individuales es diametralmente más lento que multiplicarlo por la matriz inversa previamente calculada. Lo anterior nos lleva a `QDA_Chol3`, la cual es la mejor de las tres no vectorizadas porque se logra el menor tiempo de fit: ~0.35 ms y, $1.11\times$ el cual resulta ser mejor que en QDA, gracias a `dtrtri` y lo mantiene con un tiempo de inferencia ágil de ~2.51 ms siendo este $1.24\times$ más rápido.

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
14. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

*RTA*: En el benchmarking podemos observar que TensorizedCholeskyQDA y EfficientCholeskyQDA son los que mejor resultado dan.  Si condice con lo esperado, proque al aplicar metodos de Cholesky los gastos computacionales son menores por trabajar con matrices triangulares.





## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [ ]:
%%bash
python --version
pip freeze | grep -E "scipy|numpy"

Python 3.10.22
numpy @ file:///home/conda/feedstock_root/build_artifacts/numpy_1747544640217/work/dist/numpy-2.2.6-cp310-cp310-linux_x86_64.whl#sha256=d6d964caeef85d00073d27cd62b46883d275b3d8162f723f0fcabbd0b3cc3f9d
scipy @ file:///home/conda/feedstock_root/build_artifacts/scipy-split_1739790642651/work/dist/scipy-1.15.2-cp310-cp310-linux_x86_64.whl#sha256=9e52bad6c3294d1a5b04a13632118ca2157130603c6c018c2d710162b223b27e


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [ ]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2
